# Gridded LST — seeing the real resolution

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwinkelmann/GEE_Animation/blob/prerelease/single-user-intro/notebooks/03_gridded_lst.ipynb)

A land-surface-temperature frame rendered at 720p *looks* like a 720p
measurement. It is not: Landsat's thermal band is measured at **100 m** (MODIS at
**1 km**), and everything smoother than that is upscaling cosmetics.
`render.pixel_grid` draws the native cell edges onto the frame, so the map stops
over-promising — one cell, one measurement.

This notebook is also the series' introduction to **config files** — the workflow
behind every video in `config/`. The GUI's accordions (notebook 02) and
`animate()`'s keywords are conveniences over the same keys; a YAML file is the
form you check into git, hand to a colleague, or queue twenty of overnight.
Makes live Earth Engine calls under your project.

In [ ]:
import sys
IS_COLAB = "google.colab" in sys.modules
BRANCH = "prerelease/single-user-intro"   # flipped to "main" when this PR merges
URL = f"gee_animation[shapefile] @ git+https://github.com/cwinkelmann/GEE_Animation.git@{BRANCH}"
if IS_COLAB:
    !pip install -q "{URL}"                              # deps + package (first run)
    !pip install -q --force-reinstall --no-deps "{URL}"  # branch moved? force-refresh our code
# pip resolves by version (0.2.0), which doesn't change per commit on a branch --
# --force-reinstall pulls the latest commit. After a code update: Runtime ->
# Restart session, then Run all, so the new module is re-imported.

In [ ]:
import os
PROJECT = os.environ.get("EE_PROJECT", "")   # <-- or paste your own project id here
assert PROJECT, "Set PROJECT to your own Earth Engine Cloud project id — 00_setup.ipynb shows how to create one"
AOI   = [13.75, 52.97, 13.82, 53.01]         # [west, south, east, north] — Grumsin beech forest, ~5 km
FRAME = [13.73, 52.96, 13.84, 53.02]         # the animation extent: the region grown ~1 km
START, END = "2024-06-01", "2024-09-01"      # three summer months: short on purpose

In [ ]:
from gee_animation import auth

auth.init(PROJECT)

## 1 · Two configs that differ in two keys

`pixel_grid` draws the mesh; `upscale: nearest` keeps each native cell a flat
block of one colour (the default `lanczos` would smear measurements into each
other, and the mesh would outline a fiction).

In [ ]:
import pathlib, yaml

def write_cfg(name, pixel_grid):
    # A YAML config is the EXPLICIT form: animate() fills these in for you,
    # a config file states them. That is its point — nothing implicit.
    cfg = {"name": name, "project": PROJECT,
           "start": START, "end": END, "cadence": "monthly",
           "sensor": "landsat", "index": "lst",
           "max_cloud_percent": 80,                  # scene-level pre-filter
           "aoi": {"frame": {"bbox": FRAME}, "region": {"bbox": AOI}},
           "viz": {"min": 10, "max": 40},            # deg C, fixed across frames
           "render": {"fps": 2, "scale": 30, "dimensions": 768,
                      "preset": "720p", "pixel_grid": pixel_grid,
                      "upscale": "nearest" if pixel_grid else "lanczos"},
           "out_dir": "out"}
    p = pathlib.Path(f"{name}.yaml")
    p.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    return p

cfg_plain, cfg_grid = write_cfg("lst_plain", False), write_cfg("lst_grid", True)
print(cfg_grid.read_text())

## 2 · Validate before you spend quota

`RunConfig.from_yaml` checks every key and value against the schema the whole
tool runs on — no Earth Engine involved, so it costs nothing.

In [ ]:
from gee_animation.config import RunConfig

for p in (cfg_plain, cfg_grid):
    RunConfig.from_yaml(p)          # raises ConfigError if a key is wrong
print("both valid")

Try it: delete the `preset` line from `lst_grid.yaml` and re-run the cell.
The `ConfigError` you get *is* the documentation — without an upscale the output
equals the fetched raster, every pixel is a cell edge, and the mesh would
degenerate into a flat wash. The config refuses combinations that cannot mean
anything, before a single byte is fetched.

## 3 · Render both

`cli.run` is exactly what the `gee-animation --config` command calls. One after
the other, never in parallel — Earth Engine quota is shared.

In [ ]:
from gee_animation import cli

paths_plain = cli.run(str(cfg_plain))
paths_grid = cli.run(str(cfg_grid))

## 4 · Same month, side by side

In [ ]:
import base64
from IPython.display import HTML

def last_png(paths):
    return sorted(str(p) for p in paths if str(p).endswith(".png"))[-1]

def tag(p):
    return ('<img style="width:49%" src="data:image/png;base64,'
            + base64.b64encode(open(p, "rb").read()).decode() + '">')

HTML(tag(last_png(paths_plain)) + tag(last_png(paths_grid)))

What to see, left (plain) against right (grid):

- the right frame's subtitle gained **“· 100 m grid”** — the frame declares what
  it is showing;
- inside one grid cell the colour is **constant**, because that is one
  measurement — a 100 m × 100 m average the satellite actually took;
- the smooth gradients on the left were never data. Lanczos interpolation is a
  fine way to make a presentation frame, and a poor thing to read values off.

The grid shows you the 100 m problem. The next notebook —
[04_lst_sharpening](04_lst_sharpening.ipynb) — is the method that addresses it.

## Make it yours

1. **Coarser.** Set `"sensor": "modis_lst"` and `"index": "lst_modis"` in
   `write_cfg` and watch the cells become 1 km — same forest, 100× less spatial
   information per frame.
2. **Finer.** Put the grid on `sentinel2`/`ndvi`: at 10 m the mesh nearly
   vanishes at this zoom. Resolution is relative to what you are looking at.
3. **A real config.** Open `config/r12_zoom_lst_pretty_10yr_grid.yaml` — a
   delivered site video's actual config. Change its two bboxes to *your* AOI and
   render it with `cli.run`.

*The seam behind it:* `_cap_dimensions` in `gee_animation/render.py` — read it to
see why the fetch happens at the product's native scale, which is exactly what
makes the drawn grid honest rather than decorative.